## 下面这个 cell 勿删, 导入要用到的软件包

In [1]:
import mpmath as mpm
from typing import Callable

import numpy as np
import sympy as sp

mpm.mp.dps = 50

---
## 第3题.

分别用复化梯形公式和复化 Simpson 公式计算下列积分

(2) $\displaystyle \int_0^{\pi} \dfrac{1 - \cos x}{x} ~ \mathrm{d} x,$ $m = 8$
(可看成连续函数 $\displaystyle f(x) = \begin{cases} \dfrac{1 - \cos x}{x} ~ \mathrm{d} x, & x \neq 0; \\ 0, & x = 0 \end{cases}$ 的积分);

(4) $\displaystyle \int_0^2 \dfrac{e^{-x}}{1+x^2} ~ \mathrm{d} x,$ $m = 8$.

In [2]:
def composite_trapezoid(f: Callable[[float], float], a: float, b: float, m: int = 8) -> float:
    """使用 m 个分点的复化梯形公式计算定积分 $\\int_a^b f(x) ~ \\mathrm{d}x$"""
    ### BEGIN SOLUTION
    h = (b - a) / m
    x = np.linspace(a, b, m + 1)
    y = np.array([f(xi) for xi in x])

    return h * (0.5 * y[0] + np.sum(y[1:-1]) + 0.5 * y[-1])
    ### END SOLUTION

def composite_simpson(f: Callable[[float], float], a: float, b: float, m: int = 8) -> float:
    """使用 m 个分点的复化 Simpson 公式计算定积分 $\\int_a^b f(x) ~ \\mathrm{d}x$"""
    ### BEGIN SOLUTION
    m2 = m * 2

    h = (b - a) / m2
    x = np.linspace(a, b, m2 + 1)
    y = np.array([f(xi) for xi in x])

    return h / 3 * (y[0] + 4 * np.sum(y[1:-1:2]) + 2 * np.sum(y[2:-2:2]) + y[-1])
    ### END SOLUTION

### 以下是函数定义, 勿动

In [3]:
p3_f2 = lambda x: (1 - np.cos(x)) / x if x != 0 else 0
p3_f4 = lambda x: np.exp(-x) / (1 + x**2)

In [4]:
"""Check that composite_trapezoid and composite_simpson return the correct output for problem (2)"""
mpm.mp.dps = 50

def p3_f2_mp(x):
    return (1 - mpm.cos(x)) / x if x != 0 else mpm.mpf('0')

p3_true_val_2 = mpm.quad(p3_f2_mp, [0, mpm.pi])
p3_true_val_2 = float(p3_true_val_2)

p3_approx_2 = composite_trapezoid(p3_f2, 0, np.pi, 8)
assert np.allclose(p3_approx_2, p3_true_val_2, atol=1e-2)

p3_approx_2 = composite_simpson(p3_f2, 0, np.pi, 8)
assert np.allclose(p3_approx_2, p3_true_val_2, atol=1e-4)

In [5]:
"""Check that composite_trapezoid and composite_simpson return the correct output for problem (4)"""
mpm.mp.dps = 50

def p3_f4_mp(x):
    return mpm.e**(-x) / (1 + x**2)

p3true_val_4 = mpm.quad(p3_f4_mp, [0, 2])
p3true_val_4 = float(p3true_val_4)

p3_approx_4 = composite_trapezoid(p3_f4, 0, 2, 8)
assert np.allclose(p3_approx_4, p3true_val_4, atol=1e-2)

p3_approx_4 = composite_simpson(p3_f4, 0, 2, 8)
assert np.allclose(p3_approx_4, p3true_val_4, atol=1e-4)

In [6]:
"""Hidden tests: check composite_trapezoid and composite_simpson on additional functions"""
### BEGIN HIDDEN TESTS

# Hidden function A: f(x) = exp(x) on [0, 1]  (exact = e-1)
def _h3_fa(x):
    return np.exp(x)

_h3_fa_true = np.e - 1

_h3_approx = composite_trapezoid(_h3_fa, 0, 1, 8)
assert np.allclose(_h3_approx, _h3_fa_true, atol=1e-2), \
    f"composite_trapezoid on exp(x): {_h3_approx} vs {_h3_fa_true}"

_h3_approx = composite_simpson(_h3_fa, 0, 1, 8)
assert np.allclose(_h3_approx, _h3_fa_true, atol=1e-5), \
    f"composite_simpson on exp(x): {_h3_approx} vs {_h3_fa_true}"

# Hidden function B: f(x) = sin(x) on [0, pi]  (exact = 2)
def _h3_fb(x):
    return np.sin(x)

_h3_approx = composite_trapezoid(_h3_fb, 0, np.pi, 8)
assert np.allclose(_h3_approx, 2.0, atol=5e-2), \
    f"composite_trapezoid on sin(x): {_h3_approx} vs 2.0"

_h3_approx = composite_simpson(_h3_fb, 0, np.pi, 8)
assert np.allclose(_h3_approx, 2.0, atol=1e-4), \
    f"composite_simpson on sin(x): {_h3_approx} vs 2.0"
### END HIDDEN TESTS


---

## 第4题.

用 Romberg 方法计算 $\displaystyle \int_1^2 \dfrac{\mathrm{d} x}{x},$ 精确到小数点后第8位

In [7]:
def romberg(f: Callable[[float], float], a: float, b: float, tol: float = 1e-8):
    """使用 Romberg 方法计算定积分 $\\int_a^b f(x) ~ \\mathrm{d}x$，精度为 tol"""
    ### BEGIN SOLUTION
    max_level = 10
    T = np.zeros((max_level, max_level))

    # initial trapezoid (m = 1)
    h = b - a
    T[0, 0] = 0.5 * h * (f(a) + f(b))

    for k in range(1, max_level):

        # recursive computation of composite trapezoid
        h /= 2
        m = 2 ** (k - 1)

        # new midpoints
        new_points = [a + (2 * i + 1) * h for i in range(m)]
        T[k, 0] = 0.5 * T[k - 1, 0] + h * sum(f(x) for x in new_points)

        # Romberg extrapolation
        for j in range(1, k + 1):
            T[k, j] = (4**j * T[k, j - 1] - T[k - 1, j - 1]) / (4**j - 1)

        # stopping criterion (diagonal)
        if abs(T[k, k] - T[k - 1, k - 1]) < tol:
            return T[k, k]

    # if not satisfied
    return T[max_level - 1, max_level - 1]
    ### END SOLUTION

### 以下是函数定义, 勿动

In [8]:
p4_f = lambda x: 1 / x

In [9]:
"""Check that romberg returns the correct answer"""
mpm.mp.dps = 50

p4_true_val = mpm.quad(p4_f, [1, 2])
p4_true_val = float(p4_true_val)

p4_approx = romberg(p4_f, 1, 2, 1e-8)
assert np.allclose(p4_approx, p4_true_val, atol=1e-8)

In [10]:
"""Hidden tests: check romberg on additional functions"""
### BEGIN HIDDEN TESTS

# Hidden function A: f(x) = sin(x) on [0, pi]  (exact = 2)
def _h4_fa(x):
    return np.sin(x)

_h4_approx = romberg(_h4_fa, 0, np.pi, 1e-8)
assert np.allclose(_h4_approx, 2.0, atol=1e-8), \
    f"romberg on sin(x): {_h4_approx} vs 2.0"

# Hidden function B: f(x) = exp(x) on [0, 1]  (exact = e-1)
def _h4_fb(x):
    return np.exp(x)

_h4_approx = romberg(_h4_fb, 0, 1, 1e-8)
assert np.allclose(_h4_approx, np.e - 1, atol=1e-8), \
    f"romberg on exp(x): {_h4_approx} vs {np.e - 1}"
### END HIDDEN TESTS


---
## 第5题.

用一般的积分区间上的 Gauß-Legendre 公式 (取 $n = 4$) 计算积分 $\displaystyle I(N) = \int_0^N e^{-x^2} ~ \mathrm{d} x$:

(1) $N = 1$; &nbsp;&nbsp;&nbsp;&nbsp; (2) $N = 3$; &nbsp;&nbsp;&nbsp;&nbsp; (3) $N = 10$.

并与
$$\lim_{N\to\infty} \int_0^N e^{-x^2} ~ \mathrm{d} x = \dfrac{\sqrt{\pi}}{2}$$
的结果相比较

In [11]:
def gauss_legendre(f: Callable[[float], float], a: float, b: float, n: int = 4) -> float:
    """使用 Gauss-Legendre 求积公式计算定积分 $\\int_a^b f(x) ~ \\mathrm{d}x$"""
    ### BEGIN SOLUTION
    x, w = np.polynomial.legendre.leggauss(n)
    xp = 0.5 * (b - a) * x + 0.5 * (b + a)
    wp = 0.5 * (b - a) * w

    return np.dot(wp, [f(xi) for xi in xp])
    ### END SOLUTION

### 以下是函数定义, 勿动

In [12]:
p5_f = lambda x: np.exp(-x**2)

In [14]:
"""Check that gauss_legendre returns the correct answer for N=1,3;
   for N=10 (n=4), print the result to illustrate approximation error."""
mpm.mp.dps = 50

def p5_f_mp(x):
    return mpm.e**(-x**2)

for N in [1, 3]:
    p5_true_val = float(mpm.quad(p5_f_mp, [0, N]))
    p5_approx = gauss_legendre(p5_f, 0, N, 4)
    assert np.allclose(p5_approx, p5_true_val, atol=1e-2), \
        f"N={N}: got {p5_approx}, expected {p5_true_val}"

# N=10: n=4 Gauss-Legendre is inaccurate; just display for comparison
p5_true_10 = float(mpm.quad(p5_f_mp, [0, 10]))
p5_approx_10 = gauss_legendre(p5_f, 0, 10, 4)
print(f"N=10: GL(n=4) = {p5_approx_10:.6f}, true = {p5_true_10:.6f}, error = {abs(p5_approx_10 - p5_true_10):.6f}")
print(f"√π/2 = {float(mpm.sqrt(mpm.pi)/2):.6f}")


N=10: GL(n=4) = 1.074061, true = 0.886227, error = 0.187834
√π/2 = 0.886227
